# Seeded MCMC — start from design points, stop on diagnostics

This page seeds NUTS from the best LHS points and runs until a `StopRule` is
met (split R-hat and bulk ESS). A prior-seeded run on the same budget is the
baseline.

The claim to check:

1. The **seeded** run converges in **fewer chunks** than the unseeded run under
   the same `StopRule` and chunk size.


In [ ]:
from typing import NamedTuple
import warnings

import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments, FlowModel, Property, PropertyData, PropertyMap,
    SavePlan, SaveRequest, Target, TargetSet, TransitionFlow, derived_refs,
)
from summer4.epi.calibration import BayesianModel, NormalLikelihood, Uniform, workflow as wf


## SIR design and best starts


In [ ]:
class Rates(NamedTuple):
    infection: float
    recovery: float

TRUE_INFECTION = 0.35
times = np.array([0.0, 20.0, 40.0, 60.0])
state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)
refs = derived_refs(Rates)
model = FlowModel(pmap)
model.add_flow(TransitionFlow("infection", state["S"], state["I"], refs.infection))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], refs.recovery))
cm = model.compile()
y0 = PropertyData.wrap(pmap, np.array([999.0, 1.0, 0.0]))
qty = Compartments(where=state["I"])
truth = cm.run(
    {"infection": TRUE_INFECTION, "recovery": 0.1}, y0,
    t0=0.0, t1=80.0, dt=1.0,
    save=SavePlan(requests={"I": SaveRequest(qty, ts=times)}), solver="euler",
)
obs = np.asarray(getattr(truth["I"].at_times(times).values, "data", truth["I"].at_times(times).values)).reshape(-1)
bm = BayesianModel(
    cm, {"recovery": 0.1},
    priors=(Uniform("infection", 0.05, 1.0),),
    targets=TargetSet(targets=(Target(key="I", times=times, values=obs, quantity=qty, likelihood=NormalLikelihood(sd=5.0)),)),
    y0=y0, run_kwargs={"t0": 0.0, "t1": 80.0, "dt": 1.0, "solver": "euler"},
)
design = wf.evaluate(bm, wf.lhs(bm, 64, seed=0), batch_size=32)
starts = design.best(4)


## Seeded vs unseeded NUTS

Same `StopRule(rhat=1.1, ess=40, max_samples=400)`. Plot chunks used.


In [ ]:
stop = wf.StopRule(rhat=1.1, ess=40, max_samples=400)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    seeded = wf.run_mcmc(
        bm, init=starts, num_chains=2, num_warmup=50, chunk_samples=50,
        stop=stop, jitter=0.05, seed=0,
    )
    unseeded = wf.run_mcmc(
        bm, init=None, num_chains=2, num_warmup=50, chunk_samples=50,
        stop=stop, seed=0,
    )

compare = pd.DataFrame(
    {
        "run": ["seeded", "unseeded"],
        "chunks": [seeded.chunks, unseeded.chunks],
        "converged": [seeded.converged, unseeded.converged],
    }
)
fig = compare.plot.bar(x="run", y="chunks", title="Chunks until StopRule")
fig.update_layout(xaxis_title="run", yaxis_title="chunks")
fig.show()

assert seeded.converged, seeded.reason
assert seeded.chunks <= unseeded.chunks
print(compare)
print("seeded rhat", float(seeded.diagnostics["rhat"].max()))
